# DBpedia SPARQL notebook (SPARQL-Burger)

Same queries as [`dbpedia.ipynb`](dbpedia.ipynb), built with [SPARQL-Burger](http://pmitzias.com/SPARQLBurger) instead of raw SPARQL strings.

**Important:** always call `run_query(query, ...)` — never `query.get_text()` directly. SPARQL-Burger emits invalid SPARQL for aggregates (it puts `HAVING` inside `WHERE`); `run_query` fixes that and adds matching `ORDER BY`.

| | |
|---|---|
| **Endpoint** | `https://dbpedia.org/sparql` |
| **Library** | `SPARQL-Burger` |


In [28]:
import re
import time
from typing import Any
from urllib.error import HTTPError

import polars as pl
from IPython.display import display
from SPARQLBurger.SPARQLQueryBuilder import (
    Filter,
    GroupBy,
    Having,
    Prefix,
    SPARQLGraphPattern,
    SPARQLSelectQuery,
    Triple,
)
from SPARQLWrapper import JSON, SPARQLWrapper

DBPEDIA_ENDPOINT = "https://dbpedia.org/sparql"
USER_AGENT = "SmartGridNotebook/0.1 (educational; contact: local)"

pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(-1)


def run_sparql(
    query: str,
    endpoint: str = DBPEDIA_ENDPOINT,
    retries: int = 4,
    delay_seconds: float = 3.0,
) -> pl.DataFrame:
    """Execute a SPARQL SELECT query and return bindings as a Polars DataFrame."""
    last_error: Exception | None = None

    for attempt in range(1, retries + 1):
        client = SPARQLWrapper(endpoint)
        client.setQuery(query)
        client.setReturnFormat(JSON)
        client.setTimeout(120)
        client.addCustomHttpHeader("User-Agent", USER_AGENT)

        try:
            payload: dict[str, Any] = client.query().convert()
            bindings = payload.get("results", {}).get("bindings", [])
            rows = [{key: value.get("value") for key, value in row.items()} for row in bindings]
            return pl.DataFrame(rows) if rows else pl.DataFrame()
        except HTTPError as error:
            last_error = error
            if error.code in {429, 502, 503, 504} and attempt < retries:
                print(f"DBpedia busy (HTTP {error.code}), retry {attempt}/{retries - 1}...")
                time.sleep(delay_seconds * attempt)
                continue
            raise RuntimeError(
                f"DBpedia SPARQL failed with HTTP {error.code}. "
                "The public endpoint is often overloaded — wait and retry, "
                "or paste the query at https://dbpedia.org/sparql"
            ) from error

    raise RuntimeError("DBpedia SPARQL failed after retries") from last_error


def show_df(df: pl.DataFrame) -> None:
    display(df)


def shorten_url_columns(df: pl.DataFrame) -> pl.DataFrame:
    if df.is_empty():
        return df

    shortened = df
    for column in df.columns:
        if df[column].dtype != pl.String:
            continue
        if not df[column].str.starts_with("http").any():
            continue

        shortened = shortened.with_columns(
            pl.when(pl.col(column).str.starts_with("http"))
            .then(pl.col(column).str.split("/").list.last())
            .otherwise(pl.col(column))
            .alias(column)
        )

    return shortened


def show_last_part_of_url(df: pl.DataFrame) -> None:
    show_df(shorten_url_columns(df))


def add_dbpedia_prefixes(query: SPARQLSelectQuery) -> SPARQLSelectQuery:
    query.add_prefix(Prefix("dbo", "http://dbpedia.org/ontology/"))
    query.add_prefix(Prefix("rdfs", "http://www.w3.org/2000/01/rdf-schema#"))
    return query
def finalize_query(query: SPARQLSelectQuery, *order_by: str) -> str:
    """Fix HAVING placement and append ORDER BY before LIMIT (matches dbpedia.ipynb)."""
    text = query.get_text()
    having = re.search(r"^\s+(HAVING \(.+\))$", text, re.MULTILINE)
    if having:
        clause = having.group(1)
        text = text.replace(having.group(0) + "\n", "")
        text = re.sub(r"(GROUP BY .+)", rf"\1\n{clause}", text, count=1)
    if order_by:
        text = re.sub(r"\nLIMIT ", f"\nORDER BY {' '.join(order_by)}\nLIMIT ", text, count=1)
    return text


def run_query(query: SPARQLSelectQuery, *order_by: str) -> None:
    show_last_part_of_url(run_sparql(finalize_query(query, *order_by)))



## Query 1 — 10 distinct creator names


In [29]:
pattern = SPARQLGraphPattern()
pattern.add_triples([Triple("?movie", "dbo:creator", "?creator_name")])

query = SPARQLSelectQuery(distinct=True, limit=10)
add_dbpedia_prefixes(query)
query.add_variables(["?creator_name"])
query.set_where_pattern(pattern)

run_query(query, "?creator_name")


creator_name
str
"""%22Weird_Al%22_Yankovic"""
"""&TV"""
"""12_Yard"""
"""1984_Louisiana_World_Exposition"""
"""2waytraffic"""
"""30-Second_Bunnies_Theatre"""
"""44_Blue_Productions"""
"""4chan"""
"""5-Second_Films"""


## Query 2 — Movies whose creator label starts with "a"


In [30]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?movie", "dbo:creator", "?creator_name"),
    Triple("?creator_name", "rdfs:label", "?creator_name_label"),
])
pattern.add_filter(Filter('STRSTARTS(LCASE(?creator_name_label), "a")'))

query = SPARQLSelectQuery(distinct=True, limit=10)
add_dbpedia_prefixes(query)
query.add_variables(["?movie", "?creator_name"])
query.set_where_pattern(pattern)

run_query(query, "?movie", "?creator_name")


movie,creator_name
str,str
"""'Sang_Linggo_nAPO_Sila""","""ABS-CBN"""
"""100_Days_to_Heaven""","""ABS-CBN_Studios"""
"""11er_Haus""","""Alfred_Dorfer"""
"""12_Hours_With""","""Aaron_Saidman"""
"""1992_(TV_series)""","""Alessandro_Fabbri_(screenwriter)"""
"""1993_(TV_series)""","""Alessandro_Fabbri_(screenwriter)"""
"""1994_(Italian_TV_series)""","""Alessandro_Fabbri_(screenwriter)"""
"""1DOL""","""ABS-CBN_Studios"""
"""2004:_The_Stupid_Version""","""Armando_Iannucci"""


## Query 3 — Distinct publication/publisher pairs (video game)


In [31]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?publication", "dbo:publisher", "?publisher"),
    Triple("?publication", "?publication_type", "dbo:VideoGame"),
])

query = SPARQLSelectQuery(distinct=True, limit=5)
add_dbpedia_prefixes(query)
query.add_variables(["?publication", "?publisher"])
query.set_where_pattern(pattern)

run_query(query, "?publication", "?publisher")


publication,publisher
str,str
"""'90s_Super_GP""","""Nicalis"""
"""'Splosion_Man""","""Xbox_Game_Studios"""
""".detuned""","""Sony_Interactive_Entertainment"""
"""Link""","""Bandai_Namco_Entertainment"""
"""0-D_Beat_Drop""","""Aksys_Games"""


## Query 4 — Publications with English publisher name starting with "s"


In [32]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?publication", "dbo:publisher", "?publisher"),
    Triple("?publisher", "rdfs:label", "?publisher_name"),
])
pattern.add_filter(Filter('lang(?publisher_name) = "en"'))
pattern.add_filter(Filter('STRSTARTS(LCASE(?publisher_name), "s")'))

query = SPARQLSelectQuery(distinct=True, limit=5)
add_dbpedia_prefixes(query)
query.add_variables(["?publication", "?publisher_name"])
query.set_where_pattern(pattern)

run_query(query, "?publication", "?publisher_name")


publication,publisher_name
str,str
"""%22Kodomo_o_Koroshite_Kudasai%22_to_Iu_Oya-tachi""","""Shinchosha"""
"""%22What!_Still_Alive%3F!%22""","""Syracuse University Press"""
"""'Tis_Time_for_%22Torture,%22_Princess""","""Shueisha"""
"""'t_Prinske""","""Standaard Uitgeverij"""
"""+Tic_Elder_Sister""","""Square Enix"""


## Query 5 — Video games with English publisher name starting with "a"


In [33]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?publication", "?publication_type", "dbo:VideoGame"),
    Triple("?publication", "dbo:publisher", "?publisher"),
    Triple("?publisher", "rdfs:label", "?publisher_name"),
])
pattern.add_filter(Filter('lang(?publisher_name) = "en"'))
pattern.add_filter(Filter('STRSTARTS(LCASE(?publisher_name), "a")'))

query = SPARQLSelectQuery(distinct=True, limit=5)
add_dbpedia_prefixes(query)
query.add_variables(["?publication", "?publisher_name"])
query.set_where_pattern(pattern)

run_query(query, "?publication", "?publisher_name")


publication,publisher_name
str,str
"""0-D_Beat_Drop""","""Aksys Games"""
"""0-D_Beat_Drop""","""Arc System Works"""
"""007:_Quantum_of_Solace""","""Activision"""
"""007_Legends""","""Activision"""
"""100_Bullets_(video_game)""","""Acclaim Entertainment"""


## Query 6 — Creators of video games whose English name starts with "a"


In [34]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?subject", "dbo:creator", "?creator"),
    Triple("?creator", "?made", "?video_game"),
    Triple("?video_game", "a", "dbo:VideoGame"),
    Triple("?video_game", "rdfs:label", "?video_game_name"),
])
pattern.add_filter(Filter('lang(?video_game_name) = "en"'))
pattern.add_filter(Filter('STRSTARTS(LCASE(?video_game_name), "a")'))

query = SPARQLSelectQuery(distinct=True, limit=10)
add_dbpedia_prefixes(query)
query.add_variables(["?creator", "?video_game_name"])
query.set_where_pattern(pattern)

run_query(query, "?creator", "?video_game_name")


creator,video_game_name
str,str
"""Aisling_Bea""","""Assassin's Creed III"""
"""Akiyuki_Shinbo""","""Aim for the Ace!"""
"""Alan_Tudyk""","""Alvin and the Chipmunks: Chipwrecked"""
"""Alan_Tudyk""","""Astro Boy: The Video Game"""
"""Alex_Morgan""","""Alex Hunter (character)"""
"""Alex_Ross""","""Assassin's Creed III"""
"""American_Dad!""","""Animation Throwdown: The Quest for Cards"""
"""Amy_Poehler""","""Alvin and the Chipmunks: Chipwrecked"""
"""Andrew_Stanton""","""A Bug's Life (video game)"""


## Query 7 — Video games starting with "f" that have more than one developer

Uses `HAVING` — must call `run_query(...)`, not `query.get_text()`.


In [35]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?video_game", "dbo:developer", "?developer"),
    Triple("?video_game", "a", "dbo:VideoGame"),
    Triple("?video_game", "rdfs:label", "?video_game_name"),
])
pattern.add_filter(Filter('lang(?video_game_name) = "en"'))
pattern.add_filter(Filter('STRSTARTS(LCASE(?video_game_name), "f")'))
pattern.add_having(Having("COUNT(DISTINCT ?developer) > 1"))

query = SPARQLSelectQuery(limit=10)
add_dbpedia_prefixes(query)
query.add_variables(["?video_game_name", "(COUNT(DISTINCT ?developer) AS ?developer_count)"])
query.set_where_pattern(pattern)
query.add_group_by(GroupBy(["?video_game_name"]))

run_query(query, "?video_game_name")


video_game_name,developer_count
str,str
"""F-1 World Grand Prix""","""2"""
"""F-117A Nighthawk Stealth Fighter 2.0""","""2"""
"""F-117A Stealth Fighter""","""2"""
"""F1 2000 (video game)""","""2"""
"""F1 2002 (video game)""","""2"""
"""F1 2011 (video game)""","""2"""
"""F1 2012 (video game)""","""2"""
"""F1 2013 (video game)""","""2"""
"""F1 Career Challenge""","""2"""


## Query 8 — Developers of FIFA Football 2003


In [36]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?video_game", "a", "dbo:VideoGame"),
    Triple("?video_game", "dbo:developer", "?developer"),
    Triple("?video_game", "rdfs:label", "?video_game_name"),
])
pattern.add_filter(Filter('lang(?video_game_name) = "en"'))
pattern.add_filter(Filter('STRSTARTS(LCASE(?video_game_name), "fifa football 2003")'))

query = SPARQLSelectQuery(distinct=True, limit=10)
add_dbpedia_prefixes(query)
query.add_variables(["?video_game_name", "?developer"])
query.set_where_pattern(pattern)
query.add_group_by(GroupBy(["?video_game_name"]))

run_query(query, "?video_game_name", "?developer")


video_game_name,developer
str,str
"""FIFA Football 2003""","""EA_Vancouver"""
"""FIFA Football 2003""","""Exient"""


## Query 9 — EA-published video games with release dates


In [39]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?video_game", "a", "dbo:VideoGame"),
    Triple("?video_game", "dbo:publisher", "?publisher"),
    Triple("?video_game", "rdfs:label", "?video_game_name"),
    Triple("?video_game", "dbo:releaseDate", "?release_date"),
    Triple("?publisher", "rdfs:label", "?publisher_name"),
])
pattern.add_filter(Filter('lang(?video_game_name) = "en"'))
pattern.add_filter(Filter('STRSTARTS(LCASE(?publisher_name), "ea")'))

query = SPARQLSelectQuery(distinct=True, limit=10)
add_dbpedia_prefixes(query)
query.add_variables(["?video_game_name", "?publisher", "?release_date"])
query.set_where_pattern(pattern)

run_query(query,"?release_date")


video_game_name,publisher,release_date
str,str,str
"""SimRefinery""","""Maxis""","""1992-10-26"""
"""SimAnt""","""Maxis""","""1993-02-26"""
"""FIFA International Soccer""","""EA_Sports""","""1993-12-03"""
"""NBA Showdown (video game)""","""EA_Sports""","""1994-03-29"""
"""NHL '94""","""EA_Sports""","""1994-03-31"""
"""FIFA Soccer 95""","""EA_Sports""","""1994-11-11"""
"""NHL 95""","""EA_Sports""","""1994-12-08"""
"""NBA Live 95""","""EA_Sports""","""1994-12-16"""
"""Full Tilt! Pinball""","""Maxis""","""1995-08-24"""


## Query 10 — EA-published video games released between 2001 and 2019

In [40]:
pattern = SPARQLGraphPattern()
pattern.add_triples([
    Triple("?video_game", "a", "dbo:VideoGame"),
    Triple("?video_game", "dbo:publisher", "?publisher"),
    Triple("?video_game", "rdfs:label", "?video_game_name"),
    Triple("?video_game", "dbo:releaseDate", "?release_date"),
    Triple("?publisher", "rdfs:label", "?publisher_name"),
])
pattern.add_filter(Filter('lang(?video_game_name) = "en"'))
pattern.add_filter(Filter('STRSTARTS(LCASE(?publisher_name), "ea")'))
pattern.add_filter(Filter("YEAR(?release_date) > 2000 && YEAR(?release_date) < 2020"))

query = SPARQLSelectQuery(distinct=True, limit=10)
add_dbpedia_prefixes(query)
query.add_variables(["?video_game_name", "?publisher", "?release_date"])
query.set_where_pattern(pattern)

run_query(query, "?release_date")

video_game_name,publisher,release_date
str,str,str
"""NBA Live 2001""","""EA_Sports""","""2001-01-23"""
"""NBA Live 2001""","""EA_Sports""","""2001-02-13"""
"""Triple Play Baseball""","""EA_Sports""","""2001-03-06"""
"""NHL 2001""","""EA_Sports""","""2001-03-08"""
"""Worms World Party""","""EA_Mobile""","""2001-04-06"""
"""Worms World Party""","""EA_Mobile""","""2001-04-27"""
"""NCAA Football 2002""","""EA_Sports""","""2001-07-24"""
"""FIFA Manager 09""","""EA_Sports""","""2001-09-14"""
"""FIFA Manager""","""EA_Sports""","""2001-09-14"""
